In [1]:
# Standard library imports
import os
import io
import sys
import itertools

# Third-party imports
import numpy as np
import pandas as pd
import h5py
import IPython
import PIL
import matplotlib.pyplot as plt
from tqdm import tqdm
from livelossplot import PlotLosses
from huggingface_hub import hf_hub_download

# PyTorch imports
import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split
import torchvision
from torchvision.utils import make_grid
from sklearn.model_selection import train_test_split

### Creating get targets function
For each event, separate the the targets into frames and plot them on an image grid.

**Tasks:**

- Function which takes event id and returns list of strike coodinates for each frame
    - > instantiate list outside loop, len(36)
    - > apply condition in for loop, t = event["lght"][:,0],  f = (t >= ti*5*60 - 2.5*60) & (t < ti*5*60 + 2.5*60)
    - > create 2d array to hold x and y coords 
    - > append array to list

- Function that takes these coordinates and outputs a tensor of shape (384, 384, 36)
    - > instantiate tensor of zeros shape (384, 384, 36)
    - > set each coordinate in to 1 

### Creating the dataset class
**Dataset:**
- The the tensor dataset is an itterable that should provide a single sample for each event 
- A sampe is: an input of shape (36, 4, 384, 384), and target of shape (36, 1, 384, 384)
- Initalisation:
    - Hf5 file
    - Event ids
- len():
    - len(event_ids)
- __getitem__():
    - load event from id
        - this will be a dictionary with keys: vis, ir069, ir107, vil, lght
        - >vis: (384, 384, 36) (int16)
        - >ir069: (192, 192, 36) (int16)
        - >ir107: (192, 192, 36) (int16)
        - >vil: (384, 384, 36) (uint8)
        - >lght: (38777, 5) (float32)
    - Get the four input channels and downsample them to 64x64
    - stack and reshape the four input channels to form a single input tensor to form the input: (36, 4, 64, 64)
    - normalise the input
    - Get the target and convert to image mapping using function get_targets
    - return input and target

## Utility functions

In [2]:
def load_ids(file_name="data/events.csv", frac=0.01):
    "Load ids"
    df = pd.read_csv(file_name)
    ids = df.id.unique()
    ids = np.random.choice(ids, size=int(frac*len(ids)), replace=False)
    return ids

def load_event(id):
    "Load event"
    with h5py.File(f'data/train.h5','r') as f:
        event = {img_type: f[id][img_type][:] for img_type in ['vis', 'ir069', 'ir107', 'vil', 'lght']}
    return event

def down_sample(img, to_size):
    """
    Downsample image.
    
    :param img: Input image of shape [H, W, D] where D is the number of frames.
    :param to_size: Target size for downsampling (e.g., 64 for 64x64).
    :return: Downsampled image of shape [to_size, to_size, D].
    """
    if img.ndim != 3:
        raise ValueError("Input image must have 3 dimensions [H, W, D]")
    
    img = torch.tensor(img, dtype=torch.float32)
    # Permute to shape [D, H, W] for processing
    img = img.permute(2, 0, 1).unsqueeze(1)  # Shape: [D, 1, H, W]
    downsampled_img = torch.nn.functional.interpolate(img, size=(to_size, to_size), mode='bilinear', align_corners=False)
    # Squeeze and permute back to shape [to_size, to_size, D]
    downsampled_img = downsampled_img.squeeze(1).permute(1, 2, 0).numpy()
    return downsampled_img


def normalise_sample(img):
    """
    Normalise image to [0, 1] range.
    """
    img = img - np.min(img)
    img = img / np.max(img)
    return img


def get_targets(event):
    """ 
    Maps lightning strikes of an event into a sparse target mask.
    Returned targets represent lightning strikes map for each frame (36) of the event.

    :rtype: list of sparse matrices of shape (384, 384) for each frame
    """
    t = event["lght"][:,0] 
    target = np.zeros((384, 384, 36))

    # For each frame in the event
    for ti in range(36):
        # Find which lightning strikes fall in current frame and store their coodinates
        f = (t >= ti*5*60) & (t < ti*5*60 + 5*60) 
        xs = event["lght"][f,3]
        ys = event["lght"][f,4]
        
        # Set the coordinates to 1 in the target tensor
        for x, y in zip(xs, ys):
            target[int(x), int(y), ti] += 1
            
    return target

In [139]:
max = 0
for id in ids:
    event = load_event(id)
    target = get_targets(event)
    max = np.max(target)
    if max > global_max:
        global_max = max
        global_id = id

print(global_id, global_max)

S774586 97.0


## Dataset Class

In [5]:
## Dataset Class
class LightningDataset(Dataset):
    def __init__(self, ids, target_max=100):
        self.target_max = target_max
        self.events = [load_event(id) for id in ids]

        # Downsample and normalise images
        # for event in self.events:
        #     event["vis"] = down_sample(event["vis"], to_size=64)
        #     event["vis"] = normalise_sample(event["vis"])
        #     event["vil"] = down_sample(event["vil"], to_size=64)
        #     event["vil"] = normalise_sample(event["vil"])
        #     event["ir069"] = down_sample(event["ir069"], to_size=64)
        #     event["ir069"] = normalise_sample(event["ir069"])
        #     event["ir107"] = down_sample(event["ir107"], to_size=64)

    def __len__(self):
        "Returns total number of samples"
        return len(self.events)
    
    def __getitem__(self, idx):
        "Returns a single sample from the dataset"
        # Load event from id
        event = self.events[idx]
        # Get the targets and normalise
        target = get_targets(event)
        target = target / self.target_max

        # Convert to PyTorch tensors and stack the image types into a single tensor
        vis = torch.tensor(event["vis"], dtype=torch.float32)
        vil = torch.tensor(event["vil"], dtype=torch.float32)
        ir069 = torch.tensor(event["ir069"], dtype=torch.float32)
        ir107 = torch.tensor(event["ir107"], dtype=torch.float32)

        # Stack the inputs to form a single tensor of shape (36, 4, 64, 64)
        X = torch.stack([vis, ir069, ir107, vil], dim=0).permute(3, 0, 1, 2).reshape(36*4, 386, 386)#.reshape( 36*4, 64, 64)
        # Convert dense_targets to PyTorch tensor and ensure shape is (1, 36, 384, 384)
        Y = torch.tensor(target, dtype=torch.float32).permute(2, 0, 1)

        return X, Y

In [6]:
df = pd.read_csv("data/events.csv", parse_dates=["start_utc"])
ids = df.id.unique()
id = ids[0]

In [7]:
dataset = LightningDataset([id])
print(len(dataset))
print(dataset[0][0].shape)
print(dataset[0][1].shape)

1


RuntimeError: stack expects each tensor to be equal size, but got [384, 384, 36] at entry 0 and [192, 192, 36] at entry 1

## UNet Implementation 

### Notes

**Goal:** Overfit a UNet to one batch of the dataset to prove the way we are using the data and method of learning is plausible 

**Tasks:**
- Create a UNet Model
- Input to the model is a sample of shape (36, 4, 64, 64)
- Output is a sample of shape (36, 1, 384, 384)
- The model should learn to predict the target for each frame in the event

### Defining UNet 

In [26]:
class UNet(nn.Module):
    def __init__(self, input_channels=36*4, output_channels=36):
        super(UNet, self).__init__()

        # Encoding path
        self.enc_conv1 = nn.Conv2d(input_channels, 128, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc_conv2 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = nn.Conv2d(256, 512, kernel_size=3, padding=1)

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec_conv1 = nn.Conv2d(512, 256,kernel_size=3, padding=1)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec_conv2 = nn.Conv2d(256, 128, kernel_size=3, padding=1)

        self.up3 = nn.ConvTranspose2d(128, 128, kernel_size=6, stride=6)
        self.dec_conv3 = nn.Conv2d(128, 64, kernel_size=3, padding=1)

        # Output layer
        self.output_layer = nn.Conv2d(64, output_channels, kernel_size=1)  # Linear activation (no ReLU)

    def forward(self, x):
        # Encoding path
        enc1 = torch.relu(self.enc_conv1(x))
        pool1 = self.pool1(enc1)

        enc2 = torch.relu(self.enc_conv2(pool1))
        pool2 = self.pool2(enc2)

        # Bottleneck
        bottleneck = torch.relu(self.bottleneck(pool2))

        # Decoding path using transposed convolutions
        up1 = self.up1(bottleneck)
        concat1 = torch.cat([up1, enc2], dim=1)
        dec1 = torch.relu(self.dec_conv1(concat1))

        up2 = self.up2(dec1)
        concat2 = torch.cat([up2, enc1], dim=1)
        dec2 = torch.relu(self.dec_conv2(concat2))
        
        up3 = self.up3(dec2)
        dec3 = torch.relu(self.dec_conv3(up3))

        # Output layer (applied sigmoid to ensure output is between 0 and 1)
        outputs = torch.sigmoid(self.output_layer(dec3))
        return outputs

In [ ]:
model = UNet()

# Example input tensor
batch_size = 2
temporal_frames = 36
features_per_frame = 4  # Assuming 4 features per frame
height, width = 64, 64

# Creating input in (batch, channels, height, width) format
example_input = torch.randn(batch_size, temporal_frames * features_per_frame, height, width)

# Forward pass
output = model(example_input)
print("Output shape:", output.shape)  # Expected: (1, 36, 384, 384)

: 

In [17]:
from torchvision import models
from torchsummary import summary
from torchsummary import summary


input_channels = 144
output_channels = 36
model = UNet(input_channels=input_channels, output_channels=output_channels)
summary(model, input_size=(12, 384, 384), device="cpu")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 384, 384]           6,976
         MaxPool2d-2         [-1, 64, 192, 192]               0
            Conv2d-3        [-1, 128, 192, 192]          73,856
         MaxPool2d-4          [-1, 128, 96, 96]               0
            Conv2d-5          [-1, 256, 96, 96]         295,168
   ConvTranspose2d-6        [-1, 128, 192, 192]         131,200
            Conv2d-7        [-1, 128, 192, 192]         295,040
   ConvTranspose2d-8         [-1, 64, 384, 384]          32,832
            Conv2d-9         [-1, 64, 384, 384]          73,792
           Conv2d-10         [-1, 12, 384, 384]             780
Total params: 909,644
Trainable params: 909,644
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 6.75
Forward/backward pass size (MB): 382.50
Params size (MB): 3.47
Estimated

### Model Training 

In [ ]:
from torch.utils.data import DataLoader
import random

batch_size = 10

# Load IDs and split into train and test sets
ids = load_ids(frac=0.1)
train_ids, val_ids = train_test_split(ids, test_size=0.8)

# Initialize the dataset
train_dataset = LightningDataset(train_ids, target_max=100)
val_dataset = LightningDataset(val_ids, target_max=100)

# Create DataLoaders for training and validation
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

torch.Size([5, 144, 64, 64])
torch.Size([5, 36, 384, 384])


In [ ]:
import torch.optim as optim

# Check if a GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the U-Net model and move it to the selected device
model = UNet().to(device)

# Define the loss function and optimizer
criterion = nn.MSELoss()  # Mean squared error for regression tasks
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Store loss values for plotting
train_losses = []
val_losses = []

# Training loop with loss tracking
num_epochs = 10

for epoch in range(num_epochs):
    print(f"Epoch:{epoch}")
    model.train()
    running_loss = 0.0
    for input, target in train_loader:
        input, target = input.to(device), target.to(device)
        # Forward pass
        optimizer.zero_grad()
        output = model(input)
        # Calculate loss
        loss = criterion(output, target)
        running_loss += loss.item()

        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
    # Calculate training loss
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss}")

    # Evaluate on validation set
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        for input, target in val_loader:
            input, target = input.to(device), target.to(device)

            # Forward pass
            output = model(input)
            loss = criterion(output, target)
            val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        print(f"Validation Loss: {avg_val_loss}")

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_losses, marker='o', linestyle='-', label="Training Loss")
plt.plot(range(1, num_epochs + 1), val_losses, marker='s', linestyle='-', color='red', label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss Over Epochs")
plt.grid(True)
plt.legend()
plt.show()